# 01 — Exploratory Data Analysis (EDA)

**InternHub — IT Interview Knowledge Tracing System**

Phân tích toàn bộ dữ liệu thô trước khi đưa vào pipeline KT:
- `question_bank.json` — 1,175 câu hỏi
- `virtual_users.json` — 500 người dùng giả lập
- `interaction_log.csv` — 20,000 lượt trả lời
- `self_rating_log.csv` — 2,999 lượt tự đánh giá

**Mục tiêu:** Kiểm tra chất lượng dữ liệu (null, imbalance, anomaly) và phát hiện bugs trong simulation.

## 0. Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

ON_KAGGLE = Path('/kaggle').exists()
if ON_KAGGLE:
    KG_DATASET = Path('/kaggle/input/datasets/zakhim/rcmsys/rcmsys_dataset')
    RAW        = KG_DATASET / 'data' / 'raw'
    PLOTS      = Path('/kaggle/working/plots')
else:
    ROOT  = Path('..').resolve()
    RAW   = ROOT / 'data' / 'raw'
    PLOTS = ROOT / 'notebooks' / 'plots'

PATH_QB    = RAW / 'question_bank' / 'question_bank.json'
PATH_USERS = RAW / 'simulation' / 'virtual_users.json'
PATH_LOG   = RAW / 'simulation' / 'interaction_log.csv'
PATH_SR    = RAW / 'simulation' / 'self_rating_log.csv'

PLOTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
COLORS = ['#00e5ff', '#ff7a1a', '#a3ff12', '#ff3366', '#fde047', '#c084fc']
print(f'ON_KAGGLE={ON_KAGGLE}  |  RAW={RAW}  |  PLOTS={PLOTS}')


---
## 1. Question Bank

In [ ]:
with open(PATH_QB, encoding='utf-8') as f:
    raw_qb = json.load(f)

records = []
for q in raw_qb:
    roles_raw = q.get('roles', {})
    primary = (
        roles_raw.get('primary') if isinstance(roles_raw, dict)
        else (roles_raw[0] if isinstance(roles_raw, list) and roles_raw else roles_raw)
    )
    records.append({
        'question_id':      q.get('question_id'),
        'role':             primary,
        'question_type':    q.get('question_type'),
        'difficulty_label': q.get('difficulty_label'),
        'difficulty_score': q.get('difficulty_score'),
        'skill_tags':       q.get('skill_tags', []),
        'skill_groups':     q.get('skill_groups', []),
    })

qb = pd.DataFrame(records)
print(f'Shape: {qb.shape}')
print(f'Nulls:\n{qb.isnull().sum()}')
qb.head(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

role_cnt = qb['role'].value_counts()
axes[0].bar(role_cnt.index, role_cnt.values, color=COLORS[:len(role_cnt)])
axes[0].set_title('Questions by Role')
for i, v in enumerate(role_cnt.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

diff_cnt = qb['difficulty_label'].value_counts()
axes[1].bar(diff_cnt.index, diff_cnt.values, color=COLORS[1:1+len(diff_cnt)])
axes[1].set_title('Questions by Difficulty')
for i, v in enumerate(diff_cnt.values):
    axes[1].text(i, v + 3, str(v), ha='center', fontweight='bold')

type_cnt = qb['question_type'].value_counts()
axes[2].barh(type_cnt.index, type_cnt.values, color=COLORS[2:2+len(type_cnt)])
axes[2].set_title('Questions by Type')
axes[2].set_xlabel('Count')

plt.tight_layout()


fig.savefig(PLOTS / 'qbank_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(role_cnt.to_string())
print()
print(type_cnt.to_string())

plt.tight_layout()
plt.show()

In [ ]:
# skill_groups coverage per role
sg_rows = []
for _, row in qb.iterrows():
    for sg in row['skill_groups']:
        sg_rows.append({'role': row['role'], 'skill_group': sg})

sg_df = pd.DataFrame(sg_rows)
pivot = sg_df.pivot_table(index='skill_group', columns='role', aggfunc='size', fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', ax=ax, color=COLORS[:3], width=0.7)
ax.set_title('Question Count per Skill Group x Role')
ax.set_xlabel('Skill Group')
ax.set_ylabel('# Questions')
ax.legend(title='Role')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


fig.savefig(PLOTS / 'qbank_skill_coverage.png', bbox_inches='tight', dpi=150)
plt.show()

print('Unique skill_groups in question bank:', sorted(sg_df['skill_group'].unique().tolist()))

---
## 2. Virtual Users

In [ ]:
with open(PATH_USERS, encoding='utf-8') as f:
    raw_users = json.load(f)

users = pd.DataFrame(raw_users)
print(f'Shape: {users.shape}')
print(f'Columns: {users.columns.tolist()}')
print(f'Nulls:\n{users.isnull().sum()}')
users.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

role_col = 'target_role' if 'target_role' in users.columns else 'role'
role_u = users[role_col].value_counts()
axes[0].bar(role_u.index, role_u.values, color=COLORS[:3])
axes[0].set_title('Users by Role')
for i, v in enumerate(role_u.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

type_col = 'user_type' if 'user_type' in users.columns else 'level'
if type_col in users.columns:
    type_u = users[type_col].value_counts()
    axes[1].bar(type_u.index, type_u.values, color=COLORS[3:3+len(type_u)])
    axes[1].set_title('Users by Level')
    for i, v in enumerate(type_u.values):
        axes[1].text(i, v + 1, str(v), ha='center', fontweight='bold')

plt.tight_layout()


fig.savefig(PLOTS / 'user_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

plt.tight_layout()
plt.show()

In [ ]:
# Initial skill_vector
skill_col = next((c for c in ['skill_vector', 'initial_skill', 'skills'] if c in users.columns), None)

if skill_col:
    sv_rows = []
    for _, row in users.iterrows():
        sv = row[skill_col]
        if isinstance(sv, dict):
            for k, v in sv.items():
                sv_rows.append({'skill': k, 'value': v, 'user_type': row.get('user_type', '?')})

    sv_df = pd.DataFrame(sv_rows)
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=sv_df, x='skill', y='value', hue='user_type', ax=ax, palette='muted')
    ax.set_title('Initial Skill Vector by Skill x User Type')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(PLOTS / 'user_skill_vector.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('Skill keys in user data:', sorted(sv_df['skill'].unique().tolist()))
else:
    print('No skill_vector column found in users')


---
## 3. Interaction Log

In [ ]:
log = pd.read_csv(PATH_LOG)
log['timestamp'] = pd.to_datetime(log['timestamp'], errors='coerce')

print(f'Shape: {log.shape}')
print(f'Nulls:\n{log.isnull().sum()}')
print(f'\nDtypes:\n{log.dtypes}')
log.head(3)

### 3.1 Quality Score Distribution

Quality: 0=Fail, 1=Pass HR (partial), 2=Pass Tech (full).  
Target cho beginner-dominated population: ~50% / 20% / 30%.

In [ ]:
q_cnt = log['quality_score'].value_counts().sort_index()
q_pct = log['quality_score'].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

bars = axes[0].bar(q_cnt.index.astype(str), q_cnt.values, color=COLORS[:3])
axes[0].set_title('Quality Score — Overall')
axes[0].set_xlabel('Quality (0=Fail, 1=HR Pass, 2=Tech Pass)')
for bar, pct in zip(bars, q_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{pct:.1f}%', ha='center', fontweight='bold')

role_quality = log.groupby(['role', 'quality_score']).size().unstack(fill_value=0)
role_quality_pct = role_quality.div(role_quality.sum(axis=1), axis=0) * 100
role_quality_pct.plot(kind='bar', ax=axes[1], color=COLORS[:3], width=0.7)
axes[1].set_title('Quality by Role (%)')
axes[1].set_xlabel('')
axes[1].legend(title='Quality')
plt.setp(axes[1].get_xticklabels(), rotation=0)

uid_col = 'user_id' if 'user_id' in users.columns else users.columns[0]
role_col2 = 'target_role' if 'target_role' in users.columns else 'role'
type_col2 = 'user_type' if 'user_type' in users.columns else None
if type_col2:
    log_m = log.merge(users[[uid_col, type_col2]], left_on='user_id', right_on=uid_col, how='left')
    ut_quality = log_m.groupby([type_col2, 'quality_score']).size().unstack(fill_value=0)
    ut_quality_pct = ut_quality.div(ut_quality.sum(axis=1), axis=0) * 100
    ut_quality_pct.plot(kind='bar', ax=axes[2], color=COLORS[:3], width=0.7)
    axes[2].set_title('Quality by User Type (%)')
    axes[2].set_xlabel('')
    axes[2].legend(title='Quality')
    plt.setp(axes[2].get_xticklabels(), rotation=15)

plt.tight_layout()


fig.savefig(PLOTS / 'quality_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(pd.DataFrame({'count': q_cnt, 'pct%': q_pct.round(1)}))
if q_pct.get(1, 0) < 18:
    print(f'\n⚠️  ISSUE: quality=1 chỉ {q_pct.get(1,0):.1f}% (target >=18%)')
    print('   Fix: tăng p_partial trong simulation_config.py')

plt.tight_layout()
plt.show()

### 3.2 Sequence Length per User

In [ ]:
seq_len = log.groupby('user_id').size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(seq_len, bins=30, color=COLORS[0], edgecolor='white')
axes[0].set_title('Sequence Length per User')
axes[0].set_xlabel('# Interactions')
axes[0].axvline(seq_len.mean(), color='red', linestyle='--', label=f'mean={seq_len.mean():.0f}')
axes[0].legend()

axes[1].boxplot(seq_len, vert=False, patch_artist=True,
                boxprops=dict(facecolor=COLORS[0], alpha=0.7))
axes[1].set_title('Sequence Length — Boxplot')
axes[1].set_xlabel('# Interactions')
axes[1].set_yticks([])

plt.tight_layout()


fig.savefig(PLOTS / 'sequence_length.png', bbox_inches='tight', dpi=150)
plt.show()
print(seq_len.describe().round(2))

if seq_len.std() < 1:
    print(f'\n⚠️  ISSUE: Tất cả user có đúng {int(seq_len.mean())} interactions — uniform, không thực tế')
    print('   Fix: randomize số câu hỏi per user (min 20, max 80)')

### 3.3 Learning Curve — Skill Improvement

In [ ]:
log['skill_delta'] = log['skill_after'] - log['skill_before']
lc = log.groupby('session_order')['skill_before'].mean()
delta_by_order = log.groupby('session_order')['skill_delta'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(lc.index, lc.values, color=COLORS[0], linewidth=2)
axes[0].set_title('Avg skill_before vs Session Order')
axes[0].set_xlabel('Session Order')
axes[0].set_ylabel('Avg skill_before')

bar_colors = [COLORS[2] if v >= 0 else COLORS[3] for v in delta_by_order.values]
axes[1].bar(delta_by_order.index, delta_by_order.values, color=bar_colors, alpha=0.8)
axes[1].axhline(0, color='white', linewidth=1)
axes[1].set_title('Avg skill_delta per Session (green=positive)')
axes[1].set_xlabel('Session Order')
axes[1].set_ylabel('Avg delta')

plt.tight_layout()


fig.savefig(PLOTS / 'learning_curve_eda.png', bbox_inches='tight', dpi=150)
plt.show()

total_delta = log['skill_delta'].mean()
print(f'Overall avg skill_delta: {total_delta:.4f}')
if total_delta < 0:
    print('\n⚠️  ISSUE: Avg skill delta âm — users trở nên kém hơn theo thời gian!')
    print('   Fix: sửa update rule trong simulate_interaction.py')
    print('   Expected: skill_after = skill_before + gain * quality_normalized')

plt.tight_layout()
plt.show()

In [ ]:
# Learning curve by user_type
if type_col2 and 'log_m' in dir():
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, ut in enumerate(['beginner', 'intermediate', 'advanced']):
        sub = log_m[log_m[type_col2] == ut]
        if sub.empty:
            continue
        lc_ut = sub.groupby('session_order')['skill_before'].mean()
        ax.plot(lc_ut.index, lc_ut.values, color=COLORS[i], linewidth=2, label=ut)
    ax.set_title('Learning Curve by User Type')
    ax.set_xlabel('Session Order')
    ax.set_ylabel('Avg skill_before')
    ax.legend()
    plt.tight_layout()
    

fig.savefig(PLOTS / 'learning_curve_by_type.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.4 Competency Coverage & Mismatch

In [ ]:
log_competencies = sorted(log['competency'].unique().tolist())
print(f'Competencies in interaction_log ({len(log_competencies)}):')
for c in log_competencies:
    print(' ', c)

RECOMMENDER_AXES = [
    'sql', 'analytics', 'statistics', 'visualization', 'data_engineering',
    'big_data', 'machine_learning', 'deep_learning', 'mlops', 'programming'
]
missing_in_log = set(RECOMMENDER_AXES) - set(log_competencies)
extra_in_log   = set(log_competencies) - set(RECOMMENDER_AXES)
print(f'\nIn recommender NOT in log: {sorted(missing_in_log)}')
print(f'In log NOT in recommender: {sorted(extra_in_log)}')

In [ ]:
# Quality heatmap per competency
comp_q = log.groupby('competency')['quality_score'].value_counts(normalize=True).unstack(fill_value=0) * 100

fig, ax = plt.subplots(figsize=(10, max(5, len(comp_q) * 0.45 + 2)))
sns.heatmap(comp_q, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': '%'})
ax.set_title('Quality Distribution (%) per Competency')
ax.set_ylabel('Competency')
ax.set_xlabel('Quality Score')
plt.tight_layout()


fig.savefig(PLOTS / 'quality_heatmap_competency.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Sparsity matrix
sparse = log.groupby(['user_id', 'competency']).size().unstack(fill_value=0)
n_users, n_comp = sparse.shape
nonzero = (sparse > 0).values.sum()
density = nonzero / (n_users * n_comp) * 100

print(f'User x Competency: {n_users} x {n_comp}')
print(f'Density: {density:.1f}%  ({nonzero}/{n_users*n_comp} non-zero)')

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(sparse.iloc[:50], cmap='Blues', ax=ax, linewidths=0.2,
            xticklabels=True, yticklabels=False)
ax.set_title(f'Interaction Sparsity (first 50 users) — density {density:.1f}%')
ax.set_xlabel('Competency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


fig.savefig(PLOTS / 'sparsity_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.5 Quality vs Difficulty & Skill

In [ ]:
diff_q = log.groupby('difficulty_label')['quality_score'].value_counts(normalize=True).unstack(fill_value=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

diff_q.plot(kind='bar', ax=axes[0], color=COLORS[:3], width=0.6)
axes[0].set_title('Quality (%) by Difficulty')
axes[0].set_xlabel('')
axes[0].set_ylabel('%')
axes[0].legend(title='Quality')
plt.setp(axes[0].get_xticklabels(), rotation=0)

sns.violinplot(data=log, x='quality_score', y='skill_before', palette='muted', ax=axes[1])
axes[1].set_title('skill_before distribution by Quality Score')
axes[1].set_xlabel('Quality Score')

plt.tight_layout()


fig.savefig(PLOTS / 'quality_by_difficulty.png', bbox_inches='tight', dpi=150)
plt.show()
print(diff_q.round(1))

plt.tight_layout()
plt.show()

---
## 4. Self-Rating Log

In [ ]:
sr = pd.read_csv(PATH_SR)
print(f'Shape: {sr.shape}')
print(f'Columns: {sr.columns.tolist()}')
print(f'Nulls:\n{sr.isnull().sum()}')
sr.head(3)

In [ ]:
skill_col_sr  = next((c for c in ['true_skill', 'actual_skill', 'skill'] if c in sr.columns), None)
rating_col_sr = next((c for c in ['self_rating', 'rating', 'perceived_skill'] if c in sr.columns), None)

if skill_col_sr and rating_col_sr:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(sr[skill_col_sr], sr[rating_col_sr], alpha=0.3, s=10, color=COLORS[0])
    mn, mx = sr[skill_col_sr].min(), sr[skill_col_sr].max()
    axes[0].plot([mn, mx], [mn, mx], 'r--', label='Perfect calibration')
    axes[0].set_xlabel('True Skill')
    axes[0].set_ylabel('Self Rating')
    axes[0].set_title('Self-Rating vs True Skill')
    axes[0].legend()

    sr['calibration_error'] = sr[rating_col_sr] - sr[skill_col_sr]
    axes[1].hist(sr['calibration_error'], bins=40, color=COLORS[1], edgecolor='white')
    axes[1].axvline(0, color='red', linestyle='--')
    axes[1].axvline(sr['calibration_error'].mean(), color='orange', linestyle='-.',
                    label=f'mean={sr["calibration_error"].mean():.3f}')
    axes[1].set_title('Calibration Error (Self - True)')
    axes[1].set_xlabel('Error')
    axes[1].legend()

    plt.tight_layout()
    fig.savefig(PLOTS / 'selfrating_vs_skill.png', bbox_inches='tight', dpi=150)
    plt.show()

    mean_err = sr['calibration_error'].mean()
    print(f'Mean calibration error: {mean_err:.4f}')
    if mean_err > 0.1:
        print('INFO: users tend to be over-confident (self > true)')
    elif mean_err < -0.1:
        print('INFO: users tend to be under-confident (self < true)')
else:
    print('Available columns:', sr.columns.tolist())
    print(sr.describe())


In [ ]:
if 'confidence_bias' in sr.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(sr['confidence_bias'], bins=40, color=COLORS[2], edgecolor='white')
    ax.axvline(0, color='red', linestyle='--')
    ax.axvline(sr['confidence_bias'].mean(), color='orange', linestyle='-.',
               label=f'mean={sr["confidence_bias"].mean():.3f}')
    ax.set_title('Confidence Bias Distribution')
    ax.set_xlabel('Bias (positive = over-confident)')
    ax.legend()
    plt.tight_layout()
    

fig.savefig(PLOTS / 'confidence_bias.png', bbox_inches='tight', dpi=150)
plt.show()
    print(sr['confidence_bias'].describe().round(4))

---
## 5. Tổng kết — Issues cần fix trước Feature Engineering

In [ ]:
issues = []

q1_pct = log['quality_score'].value_counts(normalize=True).get(1, 0) * 100
if q1_pct < 18:
    issues.append(('CRITICAL', 'quality=1 imbalance',
                   f'quality=1 = {q1_pct:.1f}% (target >=18%)\n'
                   '   Fix: tang p_partial trong simulation_config.py'))

avg_delta = log['skill_delta'].mean()
if avg_delta < 0:
    issues.append(('CRITICAL', 'Negative skill improvement',
                   f'avg skill_delta = {avg_delta:.4f}\n'
                   '   Fix: sua update rule trong simulate_interaction.py'))

if seq_len.std() < 1:
    issues.append(('WARNING', 'Uniform sequence length',
                   f'All users have exactly {int(seq_len.mean())} interactions\n'
                   '   Fix: randomize so cau hoi per user'))

if extra_in_log or missing_in_log:
    issues.append(('WARNING', 'Competency name mismatch',
                   f'Extra in log: {sorted(extra_in_log)}\n'
                   f'   Missing: {sorted(missing_in_log)}\n'
                   '   Fix: chuan hoa ten ve 10 axes trong recommender.py'))

print('=' * 65)
print('EDA SUMMARY')
print('=' * 65)
print(f'question_bank:  {len(raw_qb):,} questions')
print(f'virtual_users:  {len(raw_users):,} users')
print(f'interaction_log:{len(log):,} rows')
print(f'self_rating_log:{len(sr):,} rows')
print()
if not issues:
    print('No critical issues found')
for sev, title, detail in issues:
    marker = '[!]' if sev == 'CRITICAL' else '[?]'
    print(f'{marker} [{title}]')
    print(f'   {detail}')
    print()
print('=' * 65)
print('NEXT -> 02_feature_engineering.ipynb')
print('  - Fix issues above (re-run simulate_interaction.py)')
print('  - Normalize quality -> binary (>=1) for AUC + float for RMSE')
print('  - Sequence padding/truncation for DKT/SAKT')
print('  - Build cold-start feature matrix from self_rating')